In [1]:
%load_ext autoreload
%autoreload 2
"""
    Some imports used throughout the notebook
"""
import time
import cpmpy
from visualize import *

from cpmpy.transformations.normalize import toplevel_list
from cpmpy.tools.explain import make_assump_model
from factory import *
from read_data import get_data
from IPython.display import clear_output

import warnings
warnings.filterwarnings('ignore')

import numpy as np
np.set_printoptions(linewidth=90)
# preload solvers
from cpmpy import SolverLookup
names = SolverLookup.solvernames()

import networkx as nx
import re
draw = lambda g,**kwargs : nx.draw_circular(g, width=5, node_size=500,**kwargs)
cmap = ["black", "yellow", "cyan", "lightgreen", "blue"]

def graph_highlight(graph, cons, dotted=False, **kwargs):
    edges = []
    for c in cons:
        n1, n2 = c.args
        if n1.name == "max": continue
        a = int(re.search(r"\[[0-9]*\]", str(n1)).group()[1:-1])
        b = int(re.search(r"\[[0-9]*\]", str(n2)).group()[1:-1])
        edges.append((a,b))
        
    colors = ["red" if (a,b) in edges else "black" for (a,b) in graph.edges()]
    linestyles = ["dotted" if c == "red" and dotted else "solid" for c in colors]
    return draw(graph, edge_color=colors, style=linestyles, **kwargs)

In [2]:
def plot_nurserostering():
    data = get_data("Benchmarks/Instance1.txt")
    factory = NurseSchedulingFactory(data)
    model, nurse_view = factory.get_optimization_model()  # CPMpy model with all constraints

    model.solve(solver="ortools")
    return visualize(nurse_view.value(), factory)  # live decorated dataframe

## Explainable Constraint Solving

### Dr. Ignace Bleukx
#### In collaboration with dr. Dimos Tsouros and  prof. Tias Guns
#### KU Leuven, Belgium

<p>&nbsp;</p>

<table><tr style="background: white;">
    <td>&nbsp;</td>
    <td style="text-align: center; vertical-align: middle;"><img src="img/kul.jpg" width=45%></td>
    <td style="text-align: center; vertical-align: middle;"><img src="img/tuples_logo.jpeg" width=45%></td>
    <td style="text-align: center; vertical-align: middle;"><img src="img/erc.jpg" width=45%></td>
</tr></table>

<!-- Thanks to Bart Bogaerts, Emilio Gamba and Jo Devriendt -->


<small>This presentation is an executable Jupyter notebook</small>

Link to slides and more examples: https://github.com/CPMpy/XCP-explain


## Constraint Solving

<img src="img/solutions_vizual.png" width="45%" align="right" style="margin-top:100px">

Solving combinatorial optimization problems in AI

- Vehicle Routing

- Scheduling

- Manufacturing

- Other combinatorial problems ...

## Model + Solve

<center><img src="img/prob2sol.png" width=70%></center>

- What if no solution is found?
- What if the user does not _like_ the solution?
- What if the user _expected_ a different solution?
- ...


## Trustworthy & Explainable constraint solving

Human-aware AI:

- Respect human _agency_
- _Support_ users in decision making
- Provide explanations and learning opportunities

Acknowledges that a 'model' is only an approximation of the full problem,<br />
that it might result in _undesirable_ solutions.

## Explainable Constraint Programming (XCP)

In general, "**Why X?**" &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; (with X a solution or UNSAT)

<img src="img/interaction_figure4.png" width=20% align="right">

- **Deductive explanation:**
  - _What constraints cause X?_
- **Counterfactual explanation**:
  - _What to change if I want Y instead of X?_


<!-- Can be combined in interactive systems:
  - _Iteratively refine explanation & model_ -->

## Running example: Nurse Scheduling

* The assignment of _shifts_ and _holidays_ to nurses.
* Each nurse has their own restrictions and preferences,
    as does the hospital.

In [3]:
instance = "Benchmarks/Instance1.txt"  # "http://www.schedulingbenchmarks.org/nrp/data/Instance1.txt"
data = get_data(instance)
factory = NurseSchedulingFactory(data)

model, nurse_view = factory.get_decision_model()

## Nurse Rostering: data

Instances from http://www.schedulingbenchmarks.org/

"benchmark test instances from various sources including industrial collaborators and scientific publications."

<!-- 7 types of hospital constraints, 2 types of nurse constraints -->

In [4]:
plot_nurserostering()

## Hands-on Explainable Constraint Programming (XCP)


### Part 1: Explaining Satisfiability Problems
- Deductive explanations (MUS, QuickXplain, OUS, stepwise)
- Counterfactual explanations (MCS, Max-CSP, Slack-based relaxation)

### Part 2: Explaining Optimization problems
- Alternative solutions
- Counterfactual explanations (UNSAT-based, model reconcilation)

**break**

### Part 3: Explanations in an Industrial Application
- Workforce allocation and scheduling @Airbus

# Deductive explanations

_What constraints cause X?_

Explain _why_ X is a logical consequence of (part of) the input.

E.g.,
- _why_ is there no solution
- _why_ does an assignment follow from the constraints
- _..._

All these cases can typically be reduced to explaining UNSAT.

Example, explain why _fact_ follows from constraints _C_:

$\quad\quad C \models \mathit{fact} \quad\Leftrightarrow\quad C \wedge \neg \mathit{fact} \models \bot$

## Nurse rosterering, explaining unsatisfiability


<img src="img/explain_unsat.png" width="15%" align="left" style="margin:50px;">

* Model nurse rostering problem as decision problem <br>
    (no objective)
* Nurse **preferences** are also hard constraints

In [5]:
# model as decision model
factory = NurseSchedulingFactory(data)
model, nurse_view = factory.get_decision_model()  # CMPpy DECISION Model
model.solve()

False

... no solution found

In [6]:
print(f"Model has {len(model.constraints)} constraints:")
for cons in model.constraints: print("-", cons)

Model has 360 constraints:
- Megan can work at most 14 D-shifts
- Katherine can work at most 14 D-shifts
- Robert can work at most 14 D-shifts
- Jonathan can work at most 14 D-shifts
- William can work at most 14 D-shifts
- Richard can work at most 14 D-shifts
- Kristen can work at most 14 D-shifts
- Kevin can work at most 14 D-shifts
- Megan cannot work more than 4320min
- Katherine cannot work more than 4320min
- Robert cannot work more than 4320min
- Jonathan cannot work more than 4320min
- William cannot work more than 4320min
- Richard cannot work more than 4320min
- Kristen cannot work more than 4320min
- Kevin cannot work more than 4320min
- Megan should work at least 3360min
- Katherine should work at least 3360min
- Robert should work at least 3360min
- Jonathan should work at least 3360min
- William should work at least 3360min
- Richard should work at least 3360min
- Kristen should work at least 3360min
- Kevin should work at least 3360min
- Megan can work at most 5 days bef

## Deductive Explanations for UNSAT problems

The set of all constraints is unsatisfiable.

<img src="img/allcons.png" width="20%" float="right" style="margin:50px;">


But do all constraints contribute to this?

## Deductive Explanations for UNSAT problems

### Minimal Unsatisfiable Subset (MUS)

<img src="img/mus.png" width="20%" align="left" style="margin:50px;">

Pinpoint to constraints causing a conflict

... trim model to a minimal set of constraints

... minimize cognitive burden for user


### How to compute a MUS?

Deletion-based MUS algorithm

_[Joao Marques-Silva. Minimal Unsatisfiability: Models, Algorithms and Applications. ISMVL 2010. pp. 9-14]_

<img src="img/deletion_based_mus.png" width=50% align="middle">


### How to compute a MUS?

Deletion-based MUS algorithm

_[Joao Marques-Silva. Minimal Unsatisfiability: Models, Algorithms and Applications. ISMVL 2010. pp. 9-14]_


In [7]:
def mus_naive(constraints):
    m = cp.Model(constraints)
    assert m.solve() is False, "Model should be UNSAT"
    
    core = constraints
    i = 0
    while i < len(core):
        print(f"\rCurrent core size = {len(core)}, leaving out:", core[i], end="\t"*5)
        subcore = core[:i] + core[i+1:]
        if cp.Model(subcore).solve():
            i += 1 # removing makes it SAT, need to keep
        else:
            core = subcore # can safely delete 
    print()
    return core

### How to compute a MUS?

Deletion-based MUS algorithm

<img src="img/order_delmus.png" width="85%">


### How to compute a MUS, <u>efficiently</u>?

In [8]:
t0 = time.time()
core = mus_naive(model.constraints)
print(f"Naive MUS took {time.time()-t0} seconds")

Current core size = 4, leaving out: Shift D on Sun 2 must be covered by 4 nurses out of 8															
Naive MUS took 15.365546941757202 seconds


In [9]:
from cpmpy.tools.explain import mus

t0 = time.time()
core = mus(model.constraints, solver="exact")
print(f"Assumption-based MUS took {time.time()-t0} seconds")

Assumption-based MUS took 0.3599069118499756 seconds


## Assumption-based solving

Introduce indicator variable for each constraint (~on/off switch)

In [10]:
assump = cp.boolvar(shape=len(model.constraints), name="a")

cp.Model(assump.implies(model.constraints))

Constraints:
    (a[0]) -> (Megan can work at most 14 D-shifts)
    (a[1]) -> (Katherine can work at most 14 D-shifts)
    (a[2]) -> (Robert can work at most 14 D-shifts)
    (a[3]) -> (Jonathan can work at most 14 D-shifts)
    (a[4]) -> (William can work at most 14 D-shifts)
    (a[5]) -> (Richard can work at most 14 D-shifts)
    (a[6]) -> (Kristen can work at most 14 D-shifts)
    (a[7]) -> (Kevin can work at most 14 D-shifts)
    (a[8]) -> (Megan cannot work more than 4320min)
    (a[9]) -> (Katherine cannot work more than 4320min)
    (a[10]) -> (Robert cannot work more than 4320min)
    (a[11]) -> (Jonathan cannot work more than 4320min)
    (a[12]) -> (William cannot work more than 4320min)
    (a[13]) -> (Richard cannot work more than 4320min)
    (a[14]) -> (Kristen cannot work more than 4320min)
    (a[15]) -> (Kevin cannot work more than 4320min)
    (a[16]) -> (Megan should work at least 3360min)
    (a[17]) -> (Katherine should work at least 3360min)
    (a[18]) -> (Rober

### How to compute a MUS, <u>efficiently</u>?

In [ ]:
def mus_assum(constraints, solver="ortools"):
    # add indicator variable per expression
    constraints = toplevel_list(constraints, merge_and=False)
    assump = cp.boolvar(shape=len(constraints), name="assump")  # Boolean indicators
    m = cp.Model(assump.implies(constraints))  # [assump[i] -> constraints[i] for all i]
    
    s = cp.SolverLookup.get(solver, model)
    assert s.solve(assumptions=assump) is False, "Model should be UNSAT"

    core = s.get_core()  # start from solver's UNSAT core of assumption variables
    i = 0
    while i < len(core):
        subcore = core[:i] + core[i+1:]  # try all but constraint 'i'
        if s.solve(assumptions=subcore) is True:
            i += 1  # removing 'i' makes it SAT, need to keep for UNSAT
        else:
            core = subcore
    return [c for c,var in zip(constraints,assump) if var in core]

### How to compute a MUS, <u>efficiently</u>?

<center><img src="img/mus_assum.png" style="max-width: 85%;" /></center>

<img src="img/slide_cdcl1.png">

<img src="img/slide_cdcl2.png">

<img src="img/slide_cdcl3.png">

<img src="img/slide_cdcl4.png">

### How to compute a MUS, <u>efficiently</u>?

Assumption-based incremental solving **only for Boolean SAT** problems?

<b>No!</b>

Other solvers also use *clause learning* and support solving under assumptions!

* SMT solvers: <i>SAT Module Theories with CDCL</i> (e.g. Z3)
* CP-solvers: <i>Lazy Clause Generation</i> (e.g. OR-Tools, Pumpkin)
* Pseudo-Boolean solvers: <i>Conflict-Driven Cutting Plane Learning</i> (e.g. Exact)
* MaxSAT solvers: <i>Core-guided solvers</i>

### Running example

MUS explanation of why the nurse rostering instance is UNSAT


In [11]:
from cpmpy.tools.mus import mus

solver = "exact" # incremental SAT solver
subset = mus(model.constraints, solver=solver) 

print("Length of MUS:", len(subset))
for cons in subset: print("-", cons)

Length of MUS: 11
- Katherine has a day off on Sat 1
- Robert requests to work shift D on Thu 1
- Robert requests to work shift D on Tue 1
- Robert requests to work shift D on Wed 1
- Robert can work at most 5 days before having a day off
- Kevin should work at most 1 weekends
- Robert requests to work shift D on Fri 1
- Kevin requests to work shift D on Sat 2
- Shift D on Sat 1 must be covered by 5 nurses out of 8
- Richard has a day off on Sat 1
- Robert requests to work shift D on Mon 1


## Running example

MUS explanation of why the nurse rostering instance is UNSAT

In [12]:
visualize_constraints(subset, nurse_view, factory)

### Many MUS'es may exist...

_Liffiton, M.H., & Malik, A. (2013). Enumerating infeasibility: Finding multiple MUSes quickly. In
Proceedings of the 10th International Conference on Integration of AI and OR Techniques in Constraint
    Programming (CPAIOR 2013) (pp. 160–175)_

In [13]:
from cpmpy.tools.explain import marco

solver = "exact"  # incremental SAT solver
    
t0 = time.time()
cnt = 0
for (kind, sset) in marco(model.constraints, solver="exact", map_solver="pysat"):
    if kind == "MUS":
        print("M", end="")
        cnt += 1
    else: print(".", end="") # MSS
    
    if time.time() - t0 > 15:  break  # for tutorial: break after 15s
print(f"\nFound {cnt} MUSes in", time.time() - t0)


MMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMM..MMMM.MMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMM.MMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMMM
Found 445 MUSes in 15.03172492980957


### Many MUS'es may exist...

<img src="img/musses.png" width="40%" align="left" style="margin-left:50px; margin-right:50px">

This problem has just 360 constraints, yet 100.000+ MUSes exist...

Which one to show? 

Can we influence which MUS is found?

## Finding a _smallest_ or _weighted smallest_ MUS?

Give every constraint a weight: OUS: Optimal Unsatisfiable Subsets <i>(Gamba, Bogaerts, Guns, 2021)</i>.

An extension of the **implicit hitting-set approach** for finding _smallest_ MUSes <i>(Ignatiev, Janota, Marques-Silva, 2014)

<br>

**Key properties we will need**:

1. If a subset is SAT, can <i>grow</i> it to a Maximal Satisfiable Subset (MSS)
2. The complement of a MSS is a Minimal Correction Subset (MCS)
3. Theorem: A MUS is a hitting set of the MCSes (= covers at least 1 constraint in every MCS)

<table><tr>
    <td width=10%>
        <center><img width=60% src="img/mcs.png" /></center>
    </td> 
    <td width=10%>
        <center><img width=60% src="img/hittingset.png" /></center>
    </td>
</tr></table>

## Finding a _smallest_ or _weighted_smallest_ MUS?

Implicit-hitting set algorithm: **lazily** enumerate all MCSes

<table><tr>
<td width=10%>
    <center><img width=80% src="img/smus_1.png"></center>
</td>
<td width=10%>
    <center><img width=80% src="img/smus_2.png"></center>
</td>
    <td width=10%><center>...</center></td>
<td width=10%>
    <center><img width=80% src="img/smus_3.png"></center>
</td>

</tr></table>

## Optimising which MUS is found?

OUS: Optimal Unsatisfiable Subsets <i>(Gamba, Bogaerts, Guns, 2021)</i>. Every constraints has a weight.

1. Initialize sets-to-hit $\mathcal{H}$   (e.g. insert one set of all constraints)
2. Find *optimal* hitting set $S$
3. Check if SAT: grow and take complement = MCS $K$, add to sets-to-hit $\mathcal{H}$
4. Repeat until UNSAT: optimal unsatisfiable subset $S$ found

<center><img src="img/smus.png" width=55% /></center>

## <u>Efficiently</u> optimising which MUS is found?

OUS: Optimal Unsatisfiable Subsets <i>(Gamba, Bogaerts, Guns, 2021)</i>. Every constraints has a weight.

&nbsp;

<center><img src="img/smus_efficient.png" width=65%/></center>

## Running example, nurse rostering

SMUS: Smallest Unsatisfiable Subset <i>(Ignatiev et al. 2015)</i>.

In [14]:
from cpmpy.tools.explain import smus

sat_solver = "exact" # incremental SAT solver
hs_solver = "gurobi" # incremental MIP solver

small_subset = smus(model.constraints, solver=sat_solver, hs_solver=hs_solver)

print("Length of sMUS:", len(small_subset))
for cons in small_subset:  
    print("-", cons)

Length of sMUS: 3
- Robert has a day off on Tue 2
- Richard requests not to work shift D on Tue 2
- Shift D on Tue 2 must be covered by 7 nurses out of 8


In [15]:
visualize_constraints(small_subset, nurse_view, factory)

## Running example, nurse rostering

OUS: Optimal Unsatisfiable Subsets <i>(Gamba, Bogaerts, Guns, 2021)</i>. Every constraint has a weight.

In [16]:
from cpmpy.tools.explain import optimal_mus
from cpmpy.transformations.get_variables import get_variables

sat_solver = "exact" # incremental SAT solver
hs_solver = "gurobi" # incremental MIP solver

small_subset = optimal_mus(model.constraints, 
                           weights=[100 if "Robert" in str(c) else 1 for c in model.constraints],
                           solver=sat_solver, hs_solver=hs_solver)

print("Length of Optimal MUS:", len(small_subset))
for cons in small_subset:  
    print("-", cons)

Length of Optimal MUS: 8
- Megan should work at most 1 weekends
- Jonathan should work at most 1 weekends
- William should work at most 1 weekends
- Kristen should work at most 1 weekends
- Kevin should work at most 1 weekends
- Richard has a day off on Sat 1
- Shift D on Sat 1 must be covered by 5 nurses out of 8
- Shift D on Sat 2 must be covered by 6 nurses out of 8


In [17]:
visualize_constraints(small_subset, nurse_view, factory)

## Step-wise explanation

<img src="img/explain_step-wise.png" width="20%" align="right" style="margin:50px;">

- We were lucky, the SMUS is pretty understandable
- What if its not?
- Split a large MUS into smaller steps <br>
  -> Step-wise Explanations

## Step-wise explanations

Step-wise explanations originally proposed for explaining *satisfiable* problems (the unique solution or maximal consequence).

> Bogaerts, Bart, Emilio Gamba, and Tias Guns. "A framework for step-wise explaining how to solve constraint satisfaction problems." Artificial Intelligence 300 (2021): 103550.

&nbsp;

### Example: step-wise explaining a Sudoku assignment

<center><img src="img/stepwise.png" /></center>

## Step-wise explanations

**Sudoku**
- Step derives assignment
- Stop at full assignment (solution)

**UNSAT**
- Step derives invalid assignments
- Stop when domain is empty

## Step-wise explanations

- $C$: set of all constraints
- $I$: set of literals
- $f$: cost function
- $\texttt{propagate}(C)$: full propagation  

$\texttt{Explain-One-Step}$: 

find optimal MUS for each remaining literal 

($C \wedge \neg l$)

Select optimal one amongst all of them

<img src="img/stepwise_pseudo.png" width="40%" align="right" style="margin-top:-600px;">

## Step-wise explanations for any target

Risk on redundancy in sequence

<center><img src="img/stepwise.png" /></center>

Need filtering and __simplification__ in post-processing

>Ignace Bleukx, Jo Devriendt, Emilio Gamba, Bart Bogaerts, Tias Guns. Simplifying Step-wise Explanation Sequences. 29th International Conference on Principles and Practice of Constraint Programming (CP23), 2023.

### Any MUS

In [18]:
subset = mus(model.constraints)

visualize_constraints(subset, nurse_view, factory)

### Let's find a sequence of step-wise explanations of this MUS

In [19]:
from explanations.stepwise import find_sequence

seq = find_sequence(subset)

Found sequence of length 11
Filtered sequence to length 11


In [20]:
nurse_view.clear()
visualize_step(seq[0], nurse_view, factory)

Propagating constraint: Robert requests to work shift D on Mon 1


In [21]:
visualize_step(seq[1], nurse_view, factory)

Propagating constraint: Robert requests to work shift D on Tue 1


In [22]:
visualize_step(seq[2], nurse_view, factory)

Propagating constraint: Katherine has a day off on Sat 1


In [23]:
visualize_step(seq[3], nurse_view, factory)

Propagating constraint: Robert requests to work shift D on Thu 1


In [24]:
visualize_step(seq[4], nurse_view, factory)

Propagating constraint: Kevin requests to work shift D on Sat 2


In [25]:
visualize_step(seq[5], nurse_view, factory)

Propagating constraint: Kevin should work at most 1 weekends


In [27]:
visualize_step(seq[6], nurse_view, factory)

Propagating constraint: Robert requests to work shift D on Wed 1


In [28]:
visualize_step(seq[7], nurse_view, factory)

Propagating constraint: Richard has a day off on Sat 1


In [29]:
visualize_step(seq[8], nurse_view, factory)

Propagating constraint: Shift D on Sat 1 must be covered by 5 nurses out of 8


In [30]:
visualize_step(seq[9], nurse_view, factory)

Propagating constraint: Robert requests to work shift D on Fri 1


In [31]:
visualize_step(seq[10], nurse_view, factory)

Propagating constraint: Robert can work at most 5 days before having a day off


## Hands-on Explainable Constraint Programming (XCP)


### Part 1: Explaining Satisfiability Problems
- Deductive explanations (MUS, QuickXplain, OUS, stepwise)
- Counterfactual explanations (MCS, Max-CSP, Slack-based relaxation)

### Part 2: Explaining Optimization problems
- Alternative solutions
- Counterfactual explanations (UNSAT-based, model reconcilation)

**break**

### Part 3: Explanations in an Industrial Application
- Workforce allocation and scheduling @Airbus

## Counterfactual explanation of an UNSAT problem

<img src="img/fixing_mcs.png" width="20%" align="left" style="margin:50px;">

How to **change the model**, in order to find a solution?

Find subset of _soft constraints_ to **remove** <br> 

Find __optimal__ set to remove

This is textbook Max-CSP solving!

### Max-CSP solving

$\mathit{maximize} \sum w_i \cdot b_i$

$st. b_i \implies c_i$

<img src="img/mcs.png" width="40%" align="right" style="margin-top:-100px;">



In [32]:
indicators = cp.boolvar(shape=len(model.constraints), name="a")
assump_model = cp.Model(indicators.implies(model.constraints))

for c in assump_model.constraints[:10]:
    print("-",c)
print("...\n")

# satisfy as many constraints as possible
assump_model.maximize(cp.sum(indicators))
assert assump_model.solve() is True
print(f"Found optimal solution with {assump_model.objective_value()} out of {len(assump_model.constraints)} constraints")

- (a[0]) -> (Megan can work at most 14 D-shifts)
- (a[1]) -> (Katherine can work at most 14 D-shifts)
- (a[2]) -> (Robert can work at most 14 D-shifts)
- (a[3]) -> (Jonathan can work at most 14 D-shifts)
- (a[4]) -> (William can work at most 14 D-shifts)
- (a[5]) -> (Richard can work at most 14 D-shifts)
- (a[6]) -> (Kristen can work at most 14 D-shifts)
- (a[7]) -> (Kevin can work at most 14 D-shifts)
- (a[8]) -> (Megan cannot work more than 4320min)
- (a[9]) -> (Katherine cannot work more than 4320min)
...

Found optimal solution with 356 out of 360 constraints


In [33]:
print("Removed these constraints:")
for c in model.constraints:
    if c.value() is False:
        print("-", c)
        
print("Resulting solution:")
visualize(nurse_view.value(), factory)

Removed these constraints:
- Robert has a day off on Tue 2
- Richard requests not to work shift D on Tue 2
- Shift D on Sat 1 must be covered by 5 nurses out of 8
- Shift D on Sun 1 must be covered by 5 nurses out of 8
Resulting solution:


### Max-CSP solving

Finds an __optimal__ correction subset

BUT may be hard to compute (e.g., stuck proving optimality)

Instead, find non-optimal, but minimal correction subset, fast


### How to compute a Minimal Correction Subset of an UNSAT problem?

<img src="img/mcs.png" width="20%" align="left" style="margin:50px;">


Grow-based MCS/MSS

* Start from the empty (satisfiable) set

* Grow the satisable subset by adding one constraint at a time

* Each constraint that causes UNSAT is in the MCS

### Grow-based MCS

In [34]:
def mcs_naive(constraints):
    mss = []  # grow a satisfiable subset one-by-one
    mcs = []  # everything else is in the minimum conflict set
    
    for cons in constraints:
        if cp.Model(mss + [cons]).solve():
            mss.append(cons)  # adding it remains SAT
        else:
            mcs.append(cons)  # UNSAT, causes conflict
    
    return mcs

### Grow-based MCS

In [35]:
from cpmpy.tools.explain import mcs  # using assumption variables

corr_subset = mcs(model.constraints)

print("By removing these constraints, the model becomes SAT:")
for cons in corr_subset: print("-",cons)
    
visualize_constraints(corr_subset, nurse_view, factory)

By removing these constraints, the model becomes SAT:
- Robert has a day off on Tue 2
- Shift D on Sun 1 must be covered by 5 nurses out of 8
- Richard requests not to work shift D on Tue 2
- Shift D on Sat 1 must be covered by 5 nurses out of 8


In [36]:
mss = set(model.constraints) - set(corr_subset)
        
corrected_model = cp.Model(list(mss))
assert corrected_model.solve()

print(f"Counterfactual solution: ({len(corr_subset)} fewer constraints)")
visualize(nurse_view.value(), factory)

Counterfactual solution: (4 fewer constraints)


### How to compute a Minimal Correction _Subset_ of an UNSAT problem?

A minimal correction **subset** corrects an unsatisfiable problem by **dropping** constraints.

If each constraint is a clause, that is probably the best you can do...

$ $

For rich applications, like nurse rostering, there is often a softer **degree of violation**

In [37]:
visualize_constraints(mcs(model.constraints), nurse_view, factory)

E.g. from requiring 5 nurses on Saturday and Sunday to requiring 0 nurses is **pretty drastic**...

### How to compute a Minimal Correction of an UNSAT problem?

Requires defining **corrective actions**, e.g. constraint violations that allows for *relaxations* of constraints.

E.g. **feasbility restoration** by modifying rather then removing constraints

> Senthooran I, Klapperstueck M, Belov G, Czauderna T, Leo K, Wallace M, Wybrow M, Garcia de la Banda M. Human-centred __feasibility restoration__ in practice. Constraints. 2023 Jul 20:1-41.

### Types of relaxations of constraints:

<img src="img/slack.png" width="20%" align="right" style="margin:20px;">

* Boolean constraints: can only be turned on/off
* Numerical constraints: can be __violated__ to some extend
  * Introduce slack for each numerical constraint
  * Slack indicates how much a constraint may be violated
    * = fine grained penalty of solution
  * Minimize (weighted) aggregate of slack values

## Running example, nurse rostering

### Slack-based relaxations

<img src="img/slack.png" width="10%" align="right" style="margin:20px;">

E.g., allow violation of _cover constraints_ (nr of nurses on a specific day)<br>
    --> Allow days to be *slightly* under/overstaffed

In [38]:
# slack variables can only be positive here (separate over and under relaxation)
slack_under = cp.intvar(0, len(data.staff), shape=data.horizon, name="slack_under")
slack_over = cp.intvar(0, len(data.staff), shape=data.horizon, name="slack_over")

for _, cover in factory.data.cover.iterrows():
    # read the data
    day = cover["# Day"]
    shift = factory.shift_name_to_idx[cover["ShiftID"]]

    nb_nurses = cp.Count(nurse_view[:, day], shift)
    # deviation of `nb_nurses` from `requirement`
    expr = (nb_nurses == cover["Requirement"] - slack_under[day] + slack_over[day])

## Running example, nurse rostering

### Minimize global violation

In [39]:
slack_model, slack_nurse_view, slack_over, slack_under = factory.get_slack_model()  # CMPpy Model

slack = cp.cpm_array(np.append(slack_under, slack_over))
slack_model.minimize(cp.sum(slack_under) + cp.sum(slack_over)) # minimize global violation

# visualize
assert slack_model.solve()
style = visualize(slack_nurse_view.value(), factory, highlight_cover=True)
style.data.loc["Slack under"] = list(slack_under.value()) + [" "," "]
style.data.loc["Slack over"] = list(slack_over.value()) + [" ", " "]
display(style)

## Running example, nurse rostering

### Slack-based relaxations

Minimize nb of non-zero slacks constraints (similar to the MAX-CSP approach)

In [40]:
slack_model.minimize(cp.sum(slack != 0)) # minimize nb of violated constraints
 
assert slack_model.solve()
style = visualize(slack_nurse_view.value(), factory, highlight_cover=True)
style.data.loc["Slack under"] = list(slack_under.value()) + [" "]*2
style.data.loc["Slack over"] = list(slack_over.value()) + [" "]*2
display(style)

## Running example, nurse rostering

### Slack-based relaxations

Or minimize any combination

In [41]:
obj1 = cp.max(slack)       # minimize max violation
obj2 = cp.sum(slack != 0)  # minimize nb of violations
obj3 = cp.sum(slack)       # minimize global violation

slack_model.minimize(10000 * obj1 + 1000 * obj2 + obj3) # multi-objective optimization

assert slack_model.solve()
style = visualize(slack_nurse_view.value(), factory, highlight_cover=True)
style.data.loc["Slack under"] = list(slack_under.value()) + [" "]*2
style.data.loc["Slack over"] = list(slack_over.value()) + [" "]*2
display(style)

## Hands-on Explainable Constraint Programming (XCP)


### Part 1: Explaining Satisfiability Problems
- Deductive explanations (MUS, QuickXplain, OUS, stepwise)
- Counterfactual explanations (MCS, Max-CSP, Slack-based relaxation)

### Part 2: Explaining Optimization problems
- Alternative solutions
- Counterfactual explanations (UNSAT-based, model reconcilation)

**break**

### Part 3: Explanations in an Industrial Application
- Workforce allocation and scheduling @Airbus

## Explaining Optimization problems

- Model uses hard constraints and soft constraints
    - Not all constraints can be satisfied, some of them (preferences) are modeled as _soft constraints_
    - Put fine grained penalty on violation of _soft constraints_ based on their importance
- Given objective: minimize **penalty of unsatisfied preferences**
- Objective may contain subobjectives
- Subobjectives/types of preferences in nurse rostering
    - shift requests on,
    - shift requested off,
    - cover constraints

In [42]:
model, nurse_view = factory.get_optimization_model()

assert model.solve()
opt_sol = nurse_view.value()
opt_value = model.objective_value()
  
display(visualize(opt_sol, factory))
print("Total penalty:", model.objective_value())
print("Time to calculate:", model.status().runtime, "s")

Total penalty: 607
Time to calculate: 0.057323000000000006 s


## Multiple solutions

- User not satisfied with optimal solution?

- There could be multiple optimal solutions

- Find (a subset of) them by converting to a decision problem
    - Enforcing the optimal objective value

- Use `solveAll()`

In [43]:
opt_model = cp.Model(model.constraints) # init new model
opt_model += (model.objective_ == model.objective_value()) # force objective

opt_model.solveAll(solver="ortools", solution_limit=3,
                   display=lambda: display(visualize(nurse_view.value(), factory)))  # callback that visualizes sols

3

## Counterfactual explanations for optimization problems

#### What if I want Y in my solution instead of X? &#8592; Changing the solutions of our model

Our model is based on the problem definition &#8594; which is an <b>approximation of our problem</b>! <br>
\- can have undesirable solutions <br>
\- due to preferences or constraints not given to the model

  
<center><img src="img/prob2sol.png" width=70%></center>

## Counterfactual explanations for optimization problems

<img src="img/changing_solution.png" width="20%" align="right" style="margin:50px;">

#### What if I want Y in my solution instead of X? &#8592; Changing the solution
        
How to (optimally) change the solution having Y instead of X? 

<b>Y is a foil</b>: a partial assignment or constraint that is counter-factual &#8594; different from the returned solution.

Need to solve $C + Y$, with $C$ the (original) set of constraints and $Y$ the foil

* Case 1: $C + Y$ is SAT &#8594; May result in less optimal objective value <br>
\- Show new (changed) solution to the user <br>
\- Along with resulting penalty

* Case 2: $C + Y$ is UNSAT: <br>
\- counterfactual explanation of why the foil leads to UNSAT



In [44]:
# Find optimal solution
model, nurse_view = factory.get_optimization_model()

model.solve()
opt_sol = nurse_view.value()
opt_value = model.objective_value()

display(visualize(opt_sol, factory))

Robert __does not__ want to work on 1st saturday

In [45]:
Y = nurse_view[2,5] == 0 # The foil is that Robert does not want to work on 1st saturday
mmodel = model.copy()
mmodel += Y 

assert mmodel.solve()
new_penalty = mmodel.objective_value()

print("Total penalty:", new_penalty, "was:", opt_value)

Total penalty: 608 was: 607


In [46]:
style = highlight_changes(nurse_view.value(), opt_sol, factory)
display(style)

## Counterfactual explanations for optimization problems

<img src="img/changing_solution.png" width="20%" align="right" style="margin:50px;">

#### What if I want Y in my solution instead of X? &#8592; Limited changes to last solution
        
- Previous solution is very different from original! <br>
    &#8594; Do not want to change everyone's schedule!

- Change only a few parts of it?

- Tradeoff between difference and penalty



In [47]:
mmodel = model.copy()
mmodel += Y 
mmodel += mmodel.objective_value() == new_penalty # find the best solution
mmodel.minimize(cp.sum(nurse_view != opt_sol))    # using a minimum nb of changes
assert mmodel.solve()

print("Number of changes:", mmodel.objective_value())
style = highlight_changes(nurse_view.value(), opt_sol, factory)
display(style)

Number of changes: 1


## Hands-on Explainable Constraint Programming (XCP)


### Part 1: Explaining Satisfiability Problems
- Deductive explanations (MUS, QuickXplain, OUS, stepwise)
- Counterfactual explanations (MCS, Max-CSP, Slack-based relaxation)

### Part 2: Explaining Optimization problems
- Alternative solutions
- Counterfactual explanations (UNSAT-based, model reconcilation)

**break**

### Part 3: Explanations in an Industrial Application
- Workforce allocation and scheduling @Airbus

### References mentioned (many more exist!!!)

<small>
    
##### MUS
* Liffiton, M. H., & Sakallah, K. A. (2008). Algorithms for computing minimal unsatisfiable subsets of constraints. Journal of Automated Reasoning, 40, 1-33.

* Ignatiev, A., Previti, A., Liffiton, M., & Marques-Silva, J. (2015, August). Smallest MUS extraction with minimal hitting set dualization. In International Conference on Principles and Practice of Constraint Programming (pp. 173-182). Cham: Springer International Publishing.

* Joao Marques-Silva. Minimal Unsatisfiability: Models, Algorithms and Applications. ISMVL 2010. pp. 9-14

##### Feasibility restoration

* Senthooran, I., Klapperstueck, M., Belov, G., Czauderna, T., Leo, K., Wallace, M., ... & De La Banda, M. G. (2021). Human-centred feasibility restoration. In 27th International Conference on Principles and Practice of Constraint Programming (CP 2021). Schloss Dagstuhl-Leibniz-Zentrum für Informatik.

##### Explaining optimization problems
* Korikov, A., & Beck, J. C. (2021). Counterfactual explanations via inverse constraint programming. In 27th International Conference on Principles and Practice of Constraint Programming (CP 2021). Schloss Dagstuhl-Leibniz-Zentrum für Informatik.

##### Explanation in planning, ASP,  KR
* Eifler, Rebecca, Michael Cashmore, Jörg Hoffmann, Daniele Magazzeni, and Marcel Steinmetz. "A new approach to plan-space explanation: Analyzing plan-property dependencies in oversubscription planning." In Proceedings of the AAAI Conference on Artificial Intelligence, vol. 34, no. 06, pp. 9818-9826. 2020.
* Chakraborti, Tathagata, Sarath Sreedharan, Yu Zhang, and Subbarao Kambhampati. "Plan explanations as model reconciliation: moving beyond explanation as soliloquy." In Proceedings of the 26th International Joint Conference on Artificial Intelligence, pp. 156-163. 2017.
* Fandinno, Jorge, and Claudia Schulz. "Answering the “why” in answer set programming–A survey of explanation approaches." Theory and Practice of Logic Programming 19, no. 2 (2019): 114-203.
* Swartout, William, Cecile Paris, and Johanna Moore. "Explanations in knowledge systems: Design for explainable expert systems." IEEE Expert 6, no. 3 (1991): 58-64.
* Kalyanpur, Aditya, Bijan Parsia, Evren Sirin, and Bernardo Cuenca-Grau. "Repairing unsatisfiable concepts in OWL ontologies." In The Semantic Web: Research and Applications: 3rd European Semantic Web Conference, ESWC 2006 Budva, Montenegro, June 11-14, 2006 Proceedings 3, pp. 170-184. Springer Berlin Heidelberg, 2006.

#### Formal explantions in ML
* Ignatiev, Alexey, Nina Narodytska, and Joao Marques-Silva. "Abduction-based explanations for machine learning models." In Proceedings of the AAAI Conference on Artificial Intelligence, vol. 33, no. 01, pp. 1511-1519. 2019.

</small>